# CDV Phylodynamics — Dataset Exploration

Interactive companion to the pipeline scripts. This notebook **imports** the functions
from `scripts/` rather than copying them, so there is one source of truth. Fix a bug in
a script and this notebook picks it up on the next restart.

Use it for the parts that need your eyes: reviewing ambiguous records, deciding scope,
and checking that the dataset can actually support the analysis.

**Order of operations**
1. Run the fetch and curation scripts (cells below, or from the terminal)
2. Explore what came back
3. Work the `needs_review` loop until it's clean
4. Make the scope decision at the Week 3 gate
5. Hand off to alignment and tree building

> The pipeline itself lives in `scripts/`. Don't paste analysis code here permanently —
> if you write something worth keeping, move it into a script so it stays reproducible.

## Setup

**Run this section first.** Every later cell depends on the imports and on `ROOT` being
set here. If you see `NameError: name 'os' is not defined` or similar, the kernel was
restarted and this cell needs re-running — *Kernel → Restart Kernel and Run All Cells*
is the reliable way back.

In [1]:
import os, sys, subprocess, importlib.util
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Optional override. Leave as None — the cell finds the repo on its own.
REPO_ROOT = None

MARKER = Path("scripts") / "02_curate_metadata.py"

def find_repo_root(explicit=None):
    # 1. explicit override
    if explicit is not None:
        cand = Path(explicit).expanduser().resolve()
        if (cand / MARKER).is_file():
            return cand
        raise FileNotFoundError(f"REPO_ROOT is set to {cand} but {MARKER} isn't there.")

    # 2. cwd and its parents (covers notebook at repo root or in notebooks/)
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / MARKER).is_file():
            return cand

    # 3. last resort: search the home directory for it
    home = Path.home()
    hits = [p.parent.parent for p in home.rglob(str(MARKER)) if ".ipynb_checkpoints" not in str(p)]
    hits = sorted(set(hits))
    if len(hits) == 1:
        print(f"note: repo wasn't near the notebook; found it at {hits[0]}")
        return hits[0]
    if len(hits) > 1:
        raise FileNotFoundError(
            "Found more than one copy of the repo. Set REPO_ROOT to the one you want:\n"
            + "\n".join(f'    REPO_ROOT = Path("{h}")' for h in hits)
        )

    # nothing anywhere — say exactly what's wrong and what's on disk
    listing = sorted(p.name for p in cwd.iterdir())[:25] if cwd.is_dir() else []
    raise FileNotFoundError(
        f"Couldn't find scripts/02_curate_metadata.py anywhere under {home}.\n\n"
        f"cwd is {cwd}\n"
        f"cwd contains: {listing}\n\n"
        "The repo folder probably wasn't uploaded, or was uploaded flattened.\n"
        "Expected layout:\n"
        "    <repo>/scripts/    01_fetch_sequences.py, 02_curate_metadata.py, 03_align_and_tree.py\n"
        "    <repo>/config/     host_groups.tsv, vaccine_strains.txt\n"
        "    <repo>/notebooks/  this notebook\n\n"
        "Easiest fix: upload cdv-phylodynamics.zip and run in a cell:\n"
        "    import zipfile, pathlib\n"
        "    zipfile.ZipFile('cdv-phylodynamics.zip').extractall(pathlib.Path.home())"
    )

ROOT = find_repo_root(REPO_ROOT)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
print("repo root:", ROOT)
print("scripts  :", sorted(p.name for p in (ROOT / "scripts").glob("*.py")))

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.grid": True,
    "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})

RAW, INTERIM, PROCESSED = Path("data/raw"), Path("data/interim"), Path("data/processed")

repo root: /Users/batyanightingale/projects/cdv-phylodynamics
scripts  : ['01_fetch_sequences.py', '02_curate_metadata.py', '03_align_and_tree.py']


In [2]:
# Import the curation module so we can reuse its parsers interactively.
def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

curate = load_module("curate", ROOT / "scripts" / "02_curate_metadata.py")

# Sanity check that the date parser behaves — cheap insurance against a silent regression
for raw in ["2013", "Aug-2013", "14-Aug-2013", "2013-08-14", "2011/2013", "unknown"]:
    iso, dec, prec = curate.parse_collection_date(raw)
    print(f"{raw:<14} -> {str(dec):<12} ({prec})")

2013           -> 2013.5       (year)
Aug-2013       -> 2013.6219178082192 (month)
14-Aug-2013    -> 2013.6164383561643 (day)
2013-08-14     -> 2013.6164383561643 (day)
2011/2013      -> 2012.5       (range)
unknown        -> None         (none)


## Step 1 — Fetch from GenBank

Needs your email; the API key is read from the `NCBI_API_KEY` environment variable.
Start with a dry run to see how many records match before downloading anything.

In [3]:
# Your email. NCBI requires it and will contact you before blocking. Fine to keep here.
EMAIL = "batya.nightingale@gmail.com"          # <-- put your address here

# API key: read from the environment if present, otherwise prompt.
# Either way it is never written into this notebook file.
# On JupyterHub, environment variables usually aren't set, so you'll get a prompt.
# The key lives in memory for this kernel session only — re-enter after a restart.
import getpass

API_KEY = os.environ.get("NCBI_API_KEY", "")
if not API_KEY:
    API_KEY = getpass.getpass("NCBI API key (leave blank to skip): ").strip()
    if API_KEY:
        os.environ["NCBI_API_KEY"] = API_KEY   # so the subprocess scripts see it

assert EMAIL and EMAIL != "you@example.com", "Set EMAIL above before fetching."
print("email  :", EMAIL)
print("api key:", f"set ({len(API_KEY)} chars), 10 req/s" if API_KEY
      else "not set, 3 req/s")


def run_script(script, *args):
    cmd = [sys.executable, f"scripts/{script}", *map(str, args)]
    print("$", " ".join(cmd), "\n")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout or proc.stderr)
    return proc.returncode


run_script("01_fetch_sequences.py", "--email", EMAIL, "--dry-run")

NCBI API key (leave blank to skip):  ········


email  : batya.nightingale@gmail.com
api key: set (36 chars), 10 req/s
$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/01_fetch_sequences.py --email batya.nightingale@gmail.com --dry-run 

2026-08-26 19:01:15,263  INFO     using NCBI API key (10 req/s)
2026-08-26 19:01:15,263  INFO     query: txid11232[Organism:exp] AND 200:20000[Sequence Length] NOT patent[Properties]
2026-08-26 19:01:16,513  INFO     matched 3818 records
2026-08-26 19:01:16,513  INFO     dry run — exiting without download



0

In [4]:
# Full download. Skips if today's file already exists.
run_script("01_fetch_sequences.py", "--email", EMAIL)

gb_files = sorted(RAW.glob("cdv_*.gb"))
print("\nGenBank files on disk:")
for f in gb_files:
    print(f"  {f}  ({f.stat().st_size/1e6:.1f} MB)")
GB = gb_files[-1] if gb_files else None
GB

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/01_fetch_sequences.py --email batya.nightingale@gmail.com 

2026-08-26 19:01:34,208  INFO     using NCBI API key (10 req/s)
2026-08-26 19:01:34,208  INFO     query: txid11232[Organism:exp] AND 200:20000[Sequence Length] NOT patent[Properties]
2026-08-26 19:01:35,558  INFO     matched 3818 records
2026-08-26 19:01:38,890  INFO       200 / 3818 records
2026-08-26 19:01:40,134  INFO       400 / 3818 records
2026-08-26 19:01:41,714  INFO       600 / 3818 records
2026-08-26 19:01:43,990  INFO       800 / 3818 records
2026-08-26 19:01:44,974  INFO       1000 / 3818 records
2026-08-26 19:01:46,364  INFO       1200 / 3818 records
2026-08-26 19:01:51,288  INFO       1400 / 3818 records
2026-08-26 19:01:54,282  INFO       1600 / 3818 records
2026-08-26 19:01:56,398  INFO       1800 / 3818 records
2026-08-26 19:01:58,412  INFO       2000 / 3818 records
2026-08-26 19:02:00,755  INFO       2200 / 3818 records
2026-08-26 19:02:01,

PosixPath('data/raw/cdv_20260826.gb')

## Step 2 — Curate

Parses records, normalizes hosts, flags vaccine strains, parses dates, extracts H sequences.
Re-run this freely — it's deterministic and cheap.

In [5]:
assert GB is not None, "No GenBank file found. Run the fetch cell first."
run_script("02_curate_metadata.py", "--gb", GB)

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/02_curate_metadata.py --gb data/raw/cdv_20260826.gb 

2026-08-26 19:02:17,767  INFO     loaded 32 vaccine patterns, 101 host patterns
2026-08-26 19:02:19,357  INFO     parsed 3818 records
2026-08-26 19:02:19,359  INFO     excluded 88 vaccine / vaccine-derived records
2026-08-26 19:02:19,360  INFO     excluded 1350 records without an identifiable H sequence
2026-08-26 19:02:19,361  INFO     excluded 69 records with H shorter than 400 nt
2026-08-26 19:02:19,361  INFO     excluded 0 records with implausibly long H (annotation problem)
2026-08-26 19:02:19,362  INFO     excluded 412 records without a usable collection date
2026-08-26 19:02:19,363  INFO     excluded 138 records with unresolved host
2026-08-26 19:02:19,453  INFO     --------------------------------------------------------------
2026-08-26 19:02:19,453  INFO     RETAINED: 1761 of 3818 records
2026-08-26 19:02:19,453  INFO     H sequences written: 1761   (all

0

In [6]:
all_df   = pd.read_csv(INTERIM / "metadata_all.tsv", sep="\t")
clean    = pd.read_csv(PROCESSED / "metadata_clean.tsv", sep="\t")
excluded = pd.read_csv(INTERIM / "exclusions.tsv", sep="\t")
review   = pd.read_csv(INTERIM / "needs_review.tsv", sep="\t")

print(f"parsed   {len(all_df):>6}")
print(f"retained {len(clean):>6}  ({len(clean)/len(all_df):.0%})")
print(f"excluded {len(excluded):>6}")
print(f"review   {len(review):>6}")

parsed     3818
retained   1761  (46%)
excluded   2057
review      291


### Why records were excluded

This table is part of your methods section. Every number here needs a defensible reason.

In [7]:
(excluded["exclude_reason"].value_counts()
 .rename_axis("reason").reset_index(name="n")
 .assign(pct=lambda d: (100 * d.n / len(all_df)).round(1)))

,reason,n,pct
0,no_H_gene_sequence,1350,35.4
1,no_parseable_collection_date,412,10.8
2,host_unresolved,138,3.6
3,vaccine_or_vaccine_derived,88,2.3
4,H_shorter_than_400nt,69,1.8


## Step 3 — The `needs_review` loop

**This is the part that can't be automated.** For each ambiguous record: either add a
pattern to `config/host_groups.tsv`, or accept that it should be dropped.

The helper below lists every host string that failed to match, with counts, so you can
prioritise. A string appearing 40 times is worth a pattern; one appearing once may not be.

In [8]:
def unmatched_hosts(df):
    """Host strings that didn't match any pattern, most frequent first."""
    u = df[df["host_ambiguous"] & df["host_raw"].notna() & (df["host_raw"].str.strip() != "")]
    return (u["host_raw"].str.strip().value_counts()
            .rename_axis("host_raw").reset_index(name="n"))

unmatched_hosts(all_df).head(40)

,host_raw,n
0,Canis_lupus_familiaris,39
1,Paradoxurus hermaphroditus,25
2,Canis,19
3,urine,10
4,Tamandua tetradactyla,10
5,Cerdocyun thous,8
6,Canis sp.,8
7,Callithrix penicillata (Black-tufted marmoset),7
8,Vaccine,7
9,nasal swab and anticoagulant-blood,7


In [9]:
def test_pattern(pattern, df=None):
    """Preview what a candidate pattern would match BEFORE adding it to the config.

    Watch for over-matching: 'dog' matches 'raccoon dog', which is why order
    matters in host_groups.tsv. Check this output for surprises.
    """
    df = all_df if df is None else df
    hits = df[df["host_raw"].fillna("").str.lower().str.contains(pattern.lower(), regex=False)]
    print(f"'{pattern}' matches {len(hits)} records\n")
    return hits["host_raw"].value_counts().rename_axis("host_raw").reset_index(name="n")

test_pattern("fox")

'fox' matches 258 records



,host_raw,n
0,fox,184
1,red fox,16
2,Red fox (Vulpes vulpes),11
3,Urocyon cinereoargenteus (gray fox),11
4,Arctic foxes,9
5,Gray fox,5
6,Fox,4
7,Urocyon cinereoargenteus (grey fox),4
8,bat-eared fox,4
9,fox brain,2


### Ambiguous dates

Dates the parser couldn't read. Some are genuinely unusable (`unknown`, `NA`); others are
formats worth adding to `parse_collection_date()` in the curation script.

In [11]:
bad_dates = all_df[all_df["decimal_year"].isna() & all_df["collection_date_raw"].notna()]
bad_dates["collection_date_raw"].value_counts().head(25)

Series([], Name: count, dtype: int64)

### Partial H sequences found by length heuristic

These were identified by sequence length rather than annotation, so they're less certain.
Currently retained and flagged. Decide whether to keep them — more data versus more noise.

In [12]:
heur = all_df[all_df["H_source"].astype(str).str.startswith("length_heuristic")]
print(f"{len(heur)} records found by heuristic")
heur[["accession", "description", "length", "H_length", "H_source", "host_group"]].head(20)

38 records found by heuristic


,accession,description,length,H_length,H_source,host_group
183,PX224982.1,UNVERIFIED: Morbillivirus canis isolate TR-5/ similar to H protein...,1188,1188,length_heuristic_partial,domestic_dog
184,PX224981.1,UNVERIFIED: Morbillivirus canis isolate TR-18/ similar to H protei...,1188,1188,length_heuristic_partial,domestic_dog
699,OR529789.1,"Morbillivirus canis isolate Q22 nonfunctional hemagglutinin gene, ...",1824,1824,length_heuristic_full,wild_canid
1443,MT773247.1,Canine morbillivirus isolate N90 hemagglutinin protein H (gp6) gen...,738,738,length_heuristic_partial,domestic_dog
1444,MT773246.1,Canine morbillivirus isolate N62 hemagglutinin protein H (gp6) gen...,738,738,length_heuristic_partial,domestic_dog
1445,MT773245.1,Canine morbillivirus isolate N584 hemagglutinin protein H (gp6) ge...,738,738,length_heuristic_partial,domestic_dog
1446,MT773244.1,Canine morbillivirus isolate N554 hemagglutinin protein H (gp6) ge...,738,738,length_heuristic_partial,domestic_dog
1447,MT773243.1,Canine morbillivirus isolate N513 hemagglutinin protein H (gp6) ge...,738,738,length_heuristic_partial,domestic_dog
1448,MT773242.1,Canine morbillivirus isolate N481 hemagglutinin protein H (gp6) ge...,738,738,length_heuristic_partial,domestic_dog
1449,MT773241.1,Canine morbillivirus isolate N473 hemagglutinin protein H (gp6) ge...,738,738,length_heuristic_partial,domestic_dog


---
## Step 4 — The Week 3 gate

The decision: is there enough signal across enough hosts to run the analysis as scoped,
or does it need narrowing to felids only, or to one region?

In [ ]:
summary = (clean.groupby("host_group")
           .agg(n=("accession", "size"),
                earliest=("decimal_year", "min"),
                latest=("decimal_year", "max"),
                countries=("country", "nunique"),
                day_precision=("date_precision", lambda s: (s == "day").sum()))
           .sort_values("n", ascending=False))
summary["span_yrs"] = (summary["latest"] - summary["earliest"]).round(1)
summary.round(2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))

# host group composition
s = clean["host_group"].value_counts()
axes[0,0].barh(s.index[::-1], s.values[::-1], color="#4B3A79")
axes[0,0].set_title("Sequences per host group")
axes[0,0].axvline(5, color="#C0697F", ls="--", lw=1)
axes[0,0].text(5, -0.4, " n=5 floor", color="#C0697F", fontsize=8, va="top")

# sampling through time by host
for grp, sub in clean.groupby("host_group"):
    axes[0,1].scatter(sub["decimal_year"], [grp]*len(sub), s=9, alpha=0.55)
axes[0,1].set_title("Sampling through time")
axes[0,1].set_xlabel("year")

# date precision
p = clean["date_precision"].value_counts()
axes[1,0].bar(p.index, p.values, color="#6E5CA0")
axes[1,0].set_title("Date precision")

# sequences per year
yr = clean["decimal_year"].astype(int).value_counts().sort_index()
axes[1,1].bar(yr.index, yr.values, color="#4B3A79", width=0.85)
axes[1,1].set_title("Sequences per year")

plt.tight_layout()
plt.show()

**How to read these**

- **Host group counts** — the dashed line is a rough floor of 5. Groups below it can't
  support discrete trait analysis and should be merged or dropped.
- **Sampling through time** — you need temporal *spread* within groups, not just totals.
  A wild felid group where every sequence is from 2010–2012 gives BEAST almost nothing.
- **Date precision** — year-only dates are usable but add uncertainty. If most tips are
  year-only, mention it as a limitation.
- **Sequences per year** — heavy recent skew is normal and fine; a gap of a decade is not.

In [13]:
# The explicit gate check
WILD = ["wild_felid", "wild_canid", "mustelid", "procyonid", "pinniped", "ursid", "ailurid", "viverrid"]
wild = clean[clean["host_group"].isin(WILD)]

print(f"wild carnivore sequences with host + date : {len(wild)}")
print(f"distinct wild host groups (n>=5)          : {(wild['host_group'].value_counts() >= 5).sum()}")
print(f"temporal span of wild sequences           : "
      f"{wild['decimal_year'].min():.1f} - {wild['decimal_year'].max():.1f}")
print(f"domestic dog sequences                    : {(clean['host_group']=='domestic_dog').sum()}")
print()
n = len(wild)
if n >= 150:
    print("=> Comfortable. Proceed with the full cross-host analysis.")
elif n >= 60:
    print("=> Workable. Consider merging the thinner host groups.")
else:
    print("=> Thin. Narrow the scope: felids only, or one geographic region.")
    print("   Send these numbers over and we'll pick the cut together.")

wild carnivore sequences with host + date : 823
distinct wild host groups (n>=5)          : 7
temporal span of wild sequences           : 1988.5 - 2026.5
domestic dog sequences                    : 915

=> Comfortable. Proceed with the full cross-host analysis.


## Step 5 — Hand off to alignment

Once `needs_review` is worked through and the gate passes, relabel and align. The script
deliberately stops after alignment so you look at it before spending compute on a tree.

In [14]:
run_script("03_align_and_tree.py",
           "--fasta", "data/processed/sequences_H.fasta",
           "--metadata", "data/processed/metadata_clean.tsv",
           "--stop-after", "align")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/03_align_and_tree.py --fasta data/processed/sequences_H.fasta --metadata data/processed/metadata_clean.tsv --stop-after align 

2026-08-26 19:05:06,075  INFO     1761 sequences, 1761 metadata rows
2026-08-26 19:05:06,096  INFO     wrote data/processed/H_labelled.fasta (1761 sequences)
2026-08-26 19:05:06,097  INFO     wrote data/processed/H_dates.tsv
2026-08-26 19:05:06,098  INFO     host composition of the alignment set:
2026-08-26 19:05:06,098  INFO         domestic_dog       915
2026-08-26 19:05:06,098  INFO         wild_canid         386
2026-08-26 19:05:06,098  INFO         mustelid           203
2026-08-26 19:05:06,098  INFO         procyonid          144
2026-08-26 19:05:06,098  INFO         wild_felid          63
2026-08-26 19:05:06,098  INFO         other               16
2026-08-26 19:05:06,098  INFO         ailurid              9
2026-08-26 19:05:06,098  INFO         pinniped             8
2026-08-26 19:05

0

Open `data/processed/H_aligned.fasta` in **AliView** or **Jalview**. Look for ragged ends,
frame shifts, and any sequence that looks obviously misaligned. Then build the tree:

```bash
python scripts/03_align_and_tree.py \
    --fasta data/processed/sequences_H.fasta \
    --metadata data/processed/metadata_clean.tsv
```

### Then, in order

1. **Open the tree in FigTree.** Does the lineage structure look like published CDV phylogenies?
2. **Find the vaccine clade.** Any "field isolate" sitting inside it is a mislabelled vaccine
   sequence the name filter missed. Add it to `config/vaccine_strains.txt` and re-run from step 2.
   *This check is not optional — name matching provably cannot catch every vaccine-derived sequence.*
3. **Reproduce a published phylogeny** before trusting anything novel.
4. **TempEst** with `H_ml.treefile` and `H_dates.tsv` — check the root-to-tip regression.
   Positive slope and reasonable R² means tip-dated BEAST analysis is viable.

In [16]:
run_script("03_align_and_tree.py",
           "--fasta", "data/processed/sequences_H.fasta",
           "--metadata", "data/processed/metadata_clean.tsv")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/03_align_and_tree.py --fasta data/processed/sequences_H.fasta --metadata data/processed/metadata_clean.tsv 

2026-08-26 19:15:57,733  INFO     1761 sequences, 1761 metadata rows
2026-08-26 19:15:57,752  INFO     wrote data/processed/H_labelled.fasta (1761 sequences)
2026-08-26 19:15:57,753  INFO     wrote data/processed/H_dates.tsv
2026-08-26 19:15:57,754  INFO     host composition of the alignment set:
2026-08-26 19:15:57,754  INFO         domestic_dog       915
2026-08-26 19:15:57,754  INFO         wild_canid         386
2026-08-26 19:15:57,754  INFO         mustelid           203
2026-08-26 19:15:57,754  INFO         procyonid          144
2026-08-26 19:15:57,754  INFO         wild_felid          63
2026-08-26 19:15:57,754  INFO         other               16
2026-08-26 19:15:57,754  INFO         ailurid              9
2026-08-26 19:15:57,754  INFO         pinniped             8
2026-08-26 19:15:57,754  INFO      

0

In [17]:
run_script("05_tree_summary.py",
           "--tree", "data/processed/H_ml.treefile",
           "--write-table")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/05_tree_summary.py --tree data/processed/H_ml.treefile --write-table 

  1761 tips in H_ml.treefile

HOST COMPOSITION
  domestic_dog         915   52.0%
  wild_canid           386   21.9%  <- wild
  mustelid             203   11.5%  <- wild
  procyonid            144    8.2%  <- wild
  wild_felid            63    3.6%  <- wild
  other                 16    0.9%
  ailurid                9    0.5%  <- wild
  pinniped               8    0.5%  <- wild
  primate                7    0.4%
  viverrid               6    0.3%  <- wild
  ursid                  4    0.2%  <- wild

  wild carnivores total: 823 (46.7%)
  sampling years: 1982.1 - 2026.5

  MAJOR CLADES (8 groups)
  Structural split of the tree, not published lineage assignment.

  clade_1  (1241 tips)
    years: 1992.0 - 2025.7  (span 33.7)
    domestic_dog        596
    wild_canid          280  <-
    mustelid            147  <-
    procyonid           129  <-
  

0

In [18]:
run_script("05_tree_summary.py", "--tree", "data/processed/H_ml.treefile",
           "--n-clades", "20", "--write-table")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/05_tree_summary.py --tree data/processed/H_ml.treefile --n-clades 20 --write-table 

  1761 tips in H_ml.treefile

HOST COMPOSITION
  domestic_dog         915   52.0%
  wild_canid           386   21.9%  <- wild
  mustelid             203   11.5%  <- wild
  procyonid            144    8.2%  <- wild
  wild_felid            63    3.6%  <- wild
  other                 16    0.9%
  ailurid                9    0.5%  <- wild
  pinniped               8    0.5%  <- wild
  primate                7    0.4%
  viverrid               6    0.3%  <- wild
  ursid                  4    0.2%  <- wild

  wild carnivores total: 823 (46.7%)
  sampling years: 1982.1 - 2026.5

  MAJOR CLADES (20 groups)
  Structural split of the tree, not published lineage assignment.

  clade_1  (430 tips)
    years: 2004.5 - 2025.7  (span 21.2)
    domestic_dog        334
    wild_canid           50  <-
    mustelid             36  <-
    viverrid        

0

In [19]:
# global, stratified to 400
run_script("06_subsample.py", "--mode", "global", "--target", "400",
           "--aln", "data/processed/H_aligned.fasta",
           "--clades", "data/processed/H_ml_clades.tsv",
           "--metadata", "data/processed/metadata_clean.tsv")

# clade_3, the wildlife-maintained lineage — keeps all 204
run_script("06_subsample.py", "--mode", "clade", "--clade", "clade_3",
           "--aln", "data/processed/H_aligned.fasta",
           "--clades", "data/processed/H_ml_clades.tsv",
           "--metadata", "data/processed/metadata_clean.tsv")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/06_subsample.py --mode global --target 400 --aln data/processed/H_aligned.fasta --clades data/processed/H_ml_clades.tsv --metadata data/processed/metadata_clean.tsv 

Loaded 1761 aligned sequences

Pool (global): 1761 sequences
    domestic_dog       915
    wild_canid         386  <- wild
    mustelid           203  <- wild
    procyonid          144  <- wild
    wild_felid          63  <- wild
    other               16
    ailurid              9  <- wild
    pinniped             8  <- wild
    primate              7
    viverrid             6  <- wild
    ursid                4  <- wild
    wild total         823  (46.7%)
    years 1982.1 - 2026.5
    clades: clade_1=430, clade_2=409, clade_3=204, clade_4=165, clade_5=111, clade_6=109, clade_7=86, clade_8=72

Removed 332 exact duplicate sequences (kept the best-dated representative of each)

stratified subsample to 400

FINAL SUBSET: 400 sequences
    domestic_dog

0

---
## Session log

Keep notes here as you go. Decisions you make now will need justifying in the methods
section in four months, and you will not remember why.

In [20]:
import subprocess, shutil
from pathlib import Path

SUBSETS = ["H_global400", "H_clade_3"]   # add "H_clade_2", "H_clade_5" etc. as you make them
PROC = Path("data/processed")

def read_fasta(p):
    seqs, name, chunks = {}, None, []
    for line in Path(p).read_text().splitlines():
        if line.startswith(">"):
            if name: seqs[name] = "".join(chunks)
            name, chunks = line[1:].split()[0], []
        elif line.strip():
            chunks.append(line.strip())
    if name: seqs[name] = "".join(chunks)
    return seqs

def write_fasta(p, seqs, width=60):
    with Path(p).open("w") as fh:
        for n, s in seqs.items():
            fh.write(f">{n}\n")
            for i in range(0, len(s), width):
                fh.write(s[i:i+width] + "\n")

iqtree = shutil.which("iqtree2") or shutil.which("iqtree")
if not iqtree:
    raise RuntimeError("iqtree not found — conda activate cdv-phylo")

for name in SUBSETS:
    fasta = PROC / f"{name}.fasta"
    if not fasta.is_file():
        print(f"skipping {name} — {fasta} not found\n"); continue

    seqs = read_fasta(fasta)
    L = len(next(iter(seqs.values())))
    keep = [i for i in range(L) if any(s[i] not in "-." for s in seqs.values())]
    clean = PROC / f"{name}_clean.fasta"
    write_fasta(clean, {k: "".join(v[i] for i in keep) for k, v in seqs.items()})
    print(f"{name}: {len(seqs)} seqs, {L} -> {len(keep)} columns "
          f"({L - len(keep)} all-gap removed)")

    cmd = [iqtree, "-s", str(clean), "-m", "MFP", "-B", "1000",
           "--alrt", "1000", "-T", "AUTO",
           "--prefix", str(PROC / f"{name}_ml"), "-redo"]
    print("$", " ".join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-1500:] if proc.returncode == 0 else proc.stdout[-3000:] + proc.stderr[-2000:])
    print(f"--> {PROC / f'{name}_ml.treefile'}\n" + "="*60 + "\n")

H_global400: 400 seqs, 1988 -> 1987 columns (1 all-gap removed)
$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/iqtree2 -s data/processed/H_global400_clean.fasta -m MFP -B 1000 --alrt 1000 -T AUTO --prefix data/processed/H_global400_ml -redo
99  C: 0.211  G: 0.214  T: 0.276
Site proportion and rates:  (0.476,0.209) (0.452,1.412) (0.072,3.647)
Parameters optimization took 1 rounds (0.484 sec)
BEST SCORE FOUND : -32274.152

Testing tree branches by SH-like aLRT with 1000 replicates...
3.387 sec.
Creating bootstrap support values...
Split supports printed to NEXUS file data/processed/H_global400_ml.splits.nex
Total tree length: 5.088

Total number of iterations: 389
CPU time used for tree search: 4166.987 sec (1h:9m:26s)
Wall-clock time used for tree search: 1679.575 sec (0h:27m:59s)
Total CPU time used: 4190.247 sec (1h:9m:50s)
Total wall-clock time used: 1686.567 sec (0h:28m:6s)

Computing bootstrap consensus tree...
Reading input file data/processed/H_global400_ml.splits.nex...


In [21]:
import pandas as pd
from pathlib import Path

PROC = Path("data/processed")
sub = pd.read_csv(PROC / "H_clade_3_metadata.tsv", sep="\t")

# join back to the full metadata for country, strain, description
clean = pd.read_csv(PROC / "metadata_clean.tsv", sep="\t")
clean["acc_base"] = clean["accession"].astype(str).str.split(".").str[0]
cols = [c for c in ["acc_base","description","country","host_raw","host_canonical",
                    "strain","isolate","collection_date"] if c in clean.columns]
d = sub.merge(clean[cols], left_on="accession", right_on="acc_base", how="left")

WILD = {"wild_felid","wild_canid","mustelid","procyonid","pinniped",
        "ursid","ailurid","viverrid"}

print(f"clade_3: {len(d)} sequences\n")
print("HOST COMPOSITION")
h = d["host_group"].value_counts()
for host, n in h.items():
    print(f"  {host:<16} {n:>4}  {n/len(d):>6.1%}{'  <- wild' if host in WILD else ''}")
n_wild = sum(n for k, n in h.items() if k in WILD)
print(f"  {'WILD TOTAL':<16} {n_wild:>4}  {n_wild/len(d):>6.1%}")

print("\nCANONICAL HOST SPECIES (top 15)")
print(d["host_canonical"].value_counts().head(15).to_string())

print("\nGEOGRAPHY")
print(d["country"].value_counts().head(15).to_string())
print(f"  countries: {d['country'].nunique()}  |  missing: {d['country'].isna().sum()}")

print("\nHOST x DECADE")
d["decade"] = (d["decimal_year"] // 10 * 10).astype("Int64")
print(pd.crosstab(d["host_group"], d["decade"]).to_string())

print("\nDATE PRECISION")
print(d["date_precision"].value_counts().to_string())
print(f"  span: {d['decimal_year'].min():.1f} - {d['decimal_year'].max():.1f}")

print("\n" + "="*66)
print("THE 9 DOMESTIC DOGS — the most interesting tips in this clade")
print("="*66)
dogs = d[d["host_group"] == "domestic_dog"]
print(dogs[["accession","country","collection_date","strain","description"]]
      .to_string(index=False, max_colwidth=55))

unresolved = d[d["host_group"].isin(["other","unknown","unparsed"])]
if len(unresolved):
    print(f"\nUNRESOLVED HOSTS ({len(unresolved)}) — check these")
    print(unresolved[["accession","host_raw","description"]].to_string(index=False, max_colwidth=55))

clade_3: 162 sequences

HOST COMPOSITION
  procyonid          91   56.2%  <- wild
  wild_canid         32   19.8%  <- wild
  mustelid           23   14.2%  <- wild
  domestic_dog        8    4.9%
  wild_felid          7    4.3%  <- wild
  ailurid             1    0.6%  <- wild
  WILD TOTAL        154   95.1%

CANONICAL HOST SPECIES (top 15)
host_canonical
Procyon lotor               91
Canis latrans               16
Mephitis mephitis           14
Vulpes vulpes               10
Canis lupus familiaris       8
Panthera tigris              5
Neovison vison               5
Urocyon cinereoargenteus     3
Martes sp.                   3
Vulpes sp.                   2
Panthera leo                 1
Panthera pardus              1
Canis lupus                  1
Meles meles                  1
Ailurus fulgens              1

GEOGRAPHY
country
Canada     114
USA         44
Denmark      4
  countries: 3  |  missing: 0

HOST x DECADE
decade        1990  2000  2010  2020
host_group                     

In [22]:
run_script("07_make_beast_xml.py", "--aln", "data/processed/H_clade_3_clean.fasta",
           "--out", "beast/drt", "--rate", "7.46e-4",
           "--chain", "20000000", "--randomize-dates", "20")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/07_make_beast_xml.py --aln data/processed/H_clade_3_clean.fasta --out beast/drt --rate 7.46e-4 --chain 20000000 --randomize-dates 20 

162 sequences, 1950 sites
date range: 1992.00 - 2023.62 (31.6 yrs)
host groups: {'procyonid': 91, 'wild_canid': 32, 'mustelid': 23, 'domestic_dog': 8, 'wild_felid': 7, 'ailurid': 1}
rate prior mean: 0.000746   chain: 20,000,000   log every: 2,000

  Discrete trait analysis will be unreliable for these — consider merging.

wrote beast/drt/H_clade_3_clean_traits.txt   <- import this in BEAUti for the discrete trait
wrote beast/drt/H_clade_3_clean.xml   (well-formed XML)
wrote 20 date-randomised replicates -> beast/drt/date_randomised
  Run these, then compare the ucldMean 95% HPDs against the real analysis.
  Real estimate outside the randomised distribution = genuine temporal signal.

NEXT STEPS

1. Open H_clade_3_clean.xml in BEAUti (File > Load) and confirm it parses cleanly.
   This

0

In [23]:
from pathlib import Path
for d in ["data/processed", "beast", "beast/clade3"]:
    p = Path(d)
    print(f"\n{d}:", "(missing)" if not p.is_dir() else "")
    if p.is_dir():
        for f in sorted(p.iterdir()):
            print(f"   {f.name:<45} {f.stat().st_size/1024:>8.1f} KB")


data/processed: 
   .gitkeep                                           0.0 KB
   H_aligned.fasta                                 3530.0 KB
   H_clade_3.fasta                                  324.5 KB
   H_clade_3_clean.fasta                            318.3 KB
   H_clade_3_dates.tsv                                5.9 KB
   H_clade_3_metadata.tsv                            12.9 KB
   H_clade_3_ml.bionj                                 9.1 KB
   H_clade_3_ml.ckp.gz                              280.6 KB
   H_clade_3_ml.contree                               9.2 KB
   H_clade_3_ml.iqtree                               90.4 KB
   H_clade_3_ml.log                                  34.6 KB
   H_clade_3_ml.mldist                              261.4 KB
   H_clade_3_ml.model.gz                              7.2 KB
   H_clade_3_ml.splits.nex                          123.9 KB
   H_clade_3_ml.treefile                              9.7 KB
   H_dates.tsv                                       66.4 KB
   H_g

In [ ]:
# 2026-__-__
# records fetched:
# retained after curation:
# host patterns added:
# records dropped by hand, and why:
# gate decision:

In [24]:
run_script("07_make_beast_xml.py",
           "--aln", "data/processed/H_clade_3.fasta",
           "--out", "beast/clade3",
           "--rate", "7.46e-4",
           "--chain", "100000000")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/07_make_beast_xml.py --aln data/processed/H_clade_3.fasta --out beast/clade3 --rate 7.46e-4 --chain 100000000 

162 sequences, 1988 sites
date range: 1992.00 - 2023.62 (31.6 yrs)
host groups: {'procyonid': 91, 'wild_canid': 32, 'mustelid': 23, 'domestic_dog': 8, 'wild_felid': 7, 'ailurid': 1}
rate prior mean: 0.000746   chain: 100,000,000   log every: 10,000

  Discrete trait analysis will be unreliable for these — consider merging.

wrote beast/clade3/H_clade_3_traits.txt   <- import this in BEAUti for the discrete trait
wrote beast/clade3/H_clade_3.xml   (well-formed XML)

NEXT STEPS

1. Open H_clade_3.xml in BEAUti (File > Load) and confirm it parses cleanly.
   This is your check that the XML matches your BEAST version. Do it before
   committing days of compute.

2. Add the discrete trait analysis in BEAUti:
     - Install BEAST_CLASSIC via File > Manage Packages (restart BEAUti after)
     - Tip Dates tab: conf

0

In [25]:
run_script("07_make_beast_xml.py",
           "--aln", "data/processed/H_clade_3.fasta",
           "--out", "beast/clade3",
           "--rate", "7.46e-4",
           "--chain", "100000000")

$ /Users/batyanightingale/anaconda3/envs/cdv-phylo/bin/python scripts/07_make_beast_xml.py --aln data/processed/H_clade_3.fasta --out beast/clade3 --rate 7.46e-4 --chain 100000000 

162 sequences, 1988 sites
date range: 1992.00 - 2023.62 (31.6 yrs)
host groups: {'procyonid': 91, 'wild_canid': 32, 'mustelid': 23, 'domestic_dog': 8, 'wild_felid': 7, 'ailurid': 1}
rate prior mean: 0.000746   chain: 100,000,000   log every: 10,000

  Discrete trait analysis will be unreliable for these — consider merging.

wrote beast/clade3/H_clade_3_traits.txt   <- import this in BEAUti for the discrete trait
wrote beast/clade3/H_clade_3.xml   (well-formed XML)

NEXT STEPS

1. Open H_clade_3.xml in BEAUti (File > Load) and confirm it parses cleanly.
   This is your check that the XML matches your BEAST version. Do it before
   committing days of compute.

2. Add the discrete trait analysis in BEAUti:
     - Install BEAST_CLASSIC via File > Manage Packages (restart BEAUti after)
     - Tip Dates tab: conf

0

In [26]:
import xml.etree.ElementTree as ET
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # adjust to your filename
root = ET.parse(XML).getroot()
taxa = [s.get("taxon") for s in root.findall(".//sequence")]

print(f"{len(taxa)} taxa in the XML")
print("example:", repr(taxa[0]))

out = XML.parent
with (out / "traits_with_header.txt").open("w") as fh:
    fh.write("traits\thost\n")
    for t in taxa:
        fh.write(f"{t}\t{t.split('|')[1]}\n")

with (out / "traits_no_header.txt").open("w") as fh:
    for t in taxa:
        fh.write(f"{t}\t{t.split('|')[1]}\n")

print("wrote both variants to", out)
print("first data line:", repr(f"{taxa[0]}\t{taxa[0].split('|')[1]}"))

162 taxa in the XML
example: 'KU666057|procyonid|2012.896'
wrote both variants to beast/clade3
first data line: 'KU666057|procyonid|2012.896\tprocyonid'


In [27]:
import shutil, subprocess, time, re
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # <-- your edited test XML
SEED = 12345
THREADS = 4
TIMEOUT_MIN = 30        # kill it if the "test" turns into an all-nighter

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for pat in ["/Applications/BEAST*/bin/beast",
                str(Path.home() / "BEAST*/bin/beast"),
                str(Path.home() / "Applications/BEAST*/bin/beast")]:
        hits = sorted(Path(pat).parent.parent.parent.glob(Path(pat).relative_to(Path(pat).parents[2]).as_posix())) \
               if False else sorted(Path("/").glob(pat.lstrip("/")))
        if hits: return str(hits[-1])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found. Install from beast2.org, or set `beast` manually "
                       "to the full path of the executable inside BEAST*/bin/")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} not found. Files in {XML.parent}: "
                            f"{sorted(p.name for p in XML.parent.iterdir()) if XML.parent.is_dir() else 'dir missing'}")

# confirm the chain length you actually set
chain = re.search(r'chainLength="(\d+)"', XML.read_text())
print(f"beast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {int(chain.group(1)):,}" if chain else "chain : not found")
print("-" * 60)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS),
       "-overwrite", "-working", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
posteriors, tail = [], []
try:
    for line in proc.stdout:
        line = line.rstrip()
        tail.append(line); tail[:] = tail[-40:]
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
            except ValueError:
                pass
            if len(posteriors) % 10 == 1:
                print(f"  [{time.time()-t0:6.0f}s] {line[:90]}")
        elif any(k in line for k in ("Error", "Exception", "error", "Fatal")):
            print("  !!", line)
        if time.time() - t0 > TIMEOUT_MIN * 60:
            proc.kill(); print(f"\nKILLED after {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-" * 60)
print(f"exit code {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nLast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior: {posteriors[0]:.1f} -> {posteriors[-1]:.1f} "
          f"({len(posteriors)} samples)")
    print("climbing then plateauing is what you want" if posteriors[-1] > posteriors[0]
          else "posterior did not improve — check the priors")

for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>9.1f} KB")

beast : /Applications/BEAST 2.7.7/bin/beast
xml   : beast/clade3/H_clade_3.xml
chain : 100,000,000
------------------------------------------------------------
  [     1s]               0    -40116.9357    -39763.1504      -353.7852 --
  [    59s]          100000     -9473.8104     -8702.7133      -771.0970 8m52s/Msamples
  [   113s]          200000     -9309.7187     -8498.6151      -811.1036 9m2s/Msamples
  [   166s]          300000     -9367.5349     -8518.6018      -848.9330 8m55s/Msamples
  [   218s]          400000     -9371.0464     -8523.1108      -847.9355 8m50s/Msamples
  [   270s]          500000     -9303.2826     -8511.9620      -791.3206 8m49s/Msamples
  [   324s]          600000     -9320.8954     -8508.1328      -812.7625 8m50s/Msamples
  [   377s]          700000     -9373.2793     -8534.3051      -838.9742 8m51s/Msamples
  [   432s]          800000     -9365.4560     -8509.3490      -856.1069 8m53s/Msamples
  [   486s]          900000     -9352.6402     -8517.0031    

In [28]:
from pathlib import Path
import re

d = Path("beast/clade3")   # adjust to where your XML lives
logs = sorted(d.glob("*.log"))
if not logs:
    print("No .log file — BEAST likely never got past initialisation.")
else:
    f = max(logs, key=lambda p: p.stat().st_mtime)
    lines = [l for l in f.read_text().splitlines() if l and not l.startswith("#")]
    data = [l for l in lines[1:] if l.split()[0].isdigit()]
    print(f"{f.name}: {len(data)} samples logged")
    if data:
        last_state = int(data[-1].split()[0])
        mins = 30
        print(f"reached state {last_state:,} in ~{mins} min")
        print(f"rate: {last_state/mins*60:,.0f} states/hour")
        print(f"\n100M states would take {100_000_000/(last_state/mins*60):,.0f} hours "
              f"({100_000_000/(last_state/mins*60)/24:,.1f} days)")
        post = [float(l.split()[1]) for l in data]
        print(f"posterior: {post[0]:.1f} -> {post[-1]:.1f}")
        print("climbing — the chain was working, just slow" if post[-1] > post[0]
              else "flat/erratic — check priors before blaming speed")

H_clade_3.log: 1199 samples logged
reached state 11,980,000 in ~30 min
rate: 23,960,000 states/hour

100M states would take 4 hours (0.2 days)
posterior: -40116.9 -> -9300.3
climbing — the chain was working, just slow


In [29]:
import xml.etree.ElementTree as ET
from pathlib import Path

XML = Path("beast/clade3/H_clade_3.xml")   # <-- the file you actually ran

print("="*66); print(f"  {XML}"); print("="*66)
if not XML.is_file():
    d = XML.parent
    raise FileNotFoundError(f"not found. In {d}: "
        f"{sorted(p.name for p in d.iterdir()) if d.is_dir() else 'dir missing'}")

raw = XML.read_text()
root = ET.fromstring(raw)

for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")

data = root.findall(".//data")
print(f"\ndata partitions : {len(data)}")
for d in data:
    print(f"  id={d.get('id')!r:<30} sequences={len(d.findall('./sequence'))}")

print(f"\ntraitsets : {len(root.findall('.//trait'))}")
for t in root.findall(".//trait"):
    val = t.get("value") or ""
    n = len([v for v in val.split(",") if v.strip()])
    print(f"  traitname={t.get('traitname')!r:<18} entries={n:<5} "
          f"e.g. {val.split(',')[0].strip()[:45]}")

print("\nDISCRETE TRAIT CHECK")
print(f"  'host' appears               : {'host' in raw}")
print(f"  AncestralStateTreeLikelihood : {'AncestralState' in raw}")
print(f"  SVSGeneralSubstitutionModel  : {'SVSGeneralSubstitutionModel' in raw}")
print(f"  BSSVS indicators             : {'indicator' in raw.lower()}")

print("\nLOGGERS")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery'):<8} "
          f"mode={lg.get('mode') or 'params'}")

print("\nVERDICT")
has_dta = len(data) > 1 or "AncestralState" in raw or "SVSGeneralSubstitutionModel" in raw
has_hostlog = any("host" in (lg.get("fileName") or "") for lg in root.findall(".//logger"))
if not has_dta:
    print("  !! NO DISCRETE TRAIT PARTITION — you'd get a timed tree but no")
    print("     host-jump reconstruction, which is the actual result you want.")
if not has_hostlog:
    print("  !! no host.trees logger — ancestral states wouldn't be saved")
if has_dta and has_hostlog:
    print("  complete DTA analysis — ready to run")

  beast/clade3/H_clade_3.xml
chainLength : 100,000,000

data partitions : 1
  id='H_clade_3'                    sequences=162

traitsets : 1
  traitname='date-forward'     entries=162   e.g. KU666057|procyonid|2012.896=2012.8960

DISCRETE TRAIT CHECK
  'host' appears               : False
  AncestralStateTreeLikelihood : False
  SVSGeneralSubstitutionModel  : False
  BSSVS indicators             : False

LOGGERS
  H_clade_3.log                      every=10000    mode=params
  None                               every=10000    mode=params
  H_clade_3.trees                    every=10000    mode=tree

VERDICT
  !! NO DISCRETE TRAIT PARTITION — you'd get a timed tree but no
     host-jump reconstruction, which is the actual result you want.
  !! no host.trees logger — ancestral states wouldn't be saved


In [30]:
from pathlib import Path
import xml.etree.ElementTree as ET

XML = Path("beast/clade3/clade3_dta.xml")
raw = XML.read_text()
root = ET.fromstring(raw)

print(f"data partitions : {len(root.findall('.//data'))}")
for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength     : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")
print(f"BSSVS indicators: {'indicator' in raw.lower()}")
print(f"asymmetric model: {'SVSGeneralSubstitutionModel' in raw or 'AncestralState' in raw}")
print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

FileNotFoundError: [Errno 2] No such file or directory: 'beast/clade3/clade3_dta.xml'

In [31]:
from pathlib import Path
import time

hits = [p for p in Path.home().rglob("*.xml")
        if "dta" in p.name.lower() or "clade3" in p.name.lower()]
hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)

for p in hits[:10]:
    age = (time.time() - p.stat().st_mtime) / 60
    print(f"{age:7.1f} min ago  {p.stat().st_size/1024:8.1f} KB  {p}")

    2.0 min ago     365.7 KB  /Users/batyanightingale/projects/cdv-phylodynamics/data/beast:clade3:clade3_dta.xml
  291.7 min ago     353.7 KB  /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/H_clade_3_dta.xml


In [34]:
import shutil
src = Path(Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3_dta.xml)
dst = Path(Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml)
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(src), dst)
print("now at", dst.resolve())

NameError: name 'Users' is not defined

In [35]:
import shutil
from pathlib import Path

src = Path("/Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3_dta.xml")
dst = Path("/Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(src), str(dst))
print("now at", dst.resolve())

now at /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml


In [36]:
XML = Path("beast/clade3/clade3_dta.xml")


In [37]:
from pathlib import Path
import xml.etree.ElementTree as ET

XML = Path("beast/clade3/clade3_dta.xml")
raw = XML.read_text()
root = ET.fromstring(raw)

data = root.findall(".//data")
print(f"data partitions : {len(data)}")
for d in data:
    print(f"    id={d.get('id')!r:<28} sequences={len(d.findall('./sequence'))}")

for r in root.findall(".//run"):
    cl = r.get("chainLength")
    print(f"chainLength     : {int(cl):,}" if cl and cl.isdigit() else f"chainLength : {cl}")

print(f"BSSVS indicators: {'indicator' in raw.lower()}")
print(f"trait model     : {'SVSGeneralSubstitutionModel' in raw or 'AncestralState' in raw}")

print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"    {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

ok = len(data) >= 2 and any("host" in (lg.get("fileName") or "") for lg in root.findall(".//logger"))
print("\n" + ("READY — trait partition and host logger both present" if ok
              else "NOT READY — see above"))

data partitions : 2
    id='H_clade_3'                  sequences=162
    id='Host'                       sequences=0
chainLength     : 10,000,000
BSSVS indicators: True
trait model     : True

loggers:
    clade3_dta.log                     every=10000
    None                               every=1000
    clade3_dta.trees                   every=1000
    Host_tree_with_trait.trees         every=10000

NOT READY — see above


In [38]:
import re, shutil
import xml.etree.ElementTree as ET
from pathlib import Path

XML       = Path("beast/clade3/clade3_dta.xml")
CHAIN     = 10_000_000    # keep 10M; raise later only if ESS falls short
LOG_EVERY = 10_000

backup = XML.with_suffix(".xml.bak")
shutil.copy(XML, backup)
print(f"backup -> {backup.name}\n")

raw = XML.read_text()

raw, n_chain = re.subn(r'(<run\b[^>]*?chainLength=")\d+(")',
                       rf'\g<1>{CHAIN}\g<2>', raw)

def fix_logger(m):
    tag = m.group(0)
    if 'id="screenlog"' in tag:        # leave console output alone
        return tag
    return re.sub(r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', tag)

raw, n_log = re.subn(r'<logger\b[^>]*>', fix_logger, raw)
XML.write_text(raw)

# verify
root = ET.fromstring(XML.read_text())
for r in root.findall(".//run"):
    print(f"chainLength : {int(r.get('chainLength')):,}")
print("\nloggers:")
for lg in root.findall(".//logger"):
    print(f"  {str(lg.get('fileName')):<34} every={lg.get('logEvery')}")

samples = CHAIN // LOG_EVERY
print(f"\n{samples} samples per file — {'good' if samples >= 500 else 'thin, lower LOG_EVERY'}")
print(f"chain edits: {n_chain} | loggers scanned: {n_log}")

backup -> clade3_dta.xml.bak

chainLength : 10,000,000

loggers:
  clade3_dta.log                     every=10000
  None                               every=1000
  clade3_dta.trees                   every=10000
  Host_tree_with_trait.trees         every=10000

1000 samples per file — good
chain edits: 1 | loggers scanned: 4


In [39]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta.xml")
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 90

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found. Set `beast` to the full path of BEAST*/bin/beast")
if not XML.is_file():
    d = XML.parent
    raise FileNotFoundError(f"{XML} missing. In {d}: "
        f"{sorted(p.name for p in d.iterdir()) if d.is_dir() else 'dir missing'}")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
print(f"beast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {int(chain.group(1)):,}" if chain else "chain : ?")
print(f"seed  : {SEED}")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=XML.parent, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, n = [], [], 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if any(k in low for k in ("state", "trait", "beagle", "error", "exception")) and n < 25:
            print(f"  {line[:100]}"); n += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1:
                    el = time.time() - t0
                    frac = int(parts[0]) / int(chain.group(1)) if chain else 0
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    flag = "  <-- EMPTY" if kb < 1 else ""
    print(f"  {f.name:<38} {kb:>10.1f} KB{flag}")

beast : /Applications/BEAST 2.7.7/bin/beast
xml   : beast/clade3/clade3_dta.xml
chain : 10,000,000
seed  : 12345
------------------------------------------------------------------
  java.io.FileNotFoundException: /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/beast
------------------------------------------------------------------
exit 1 after 0.0 min

last lines:
  java.io.FileNotFoundException: /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/beast/clade3/clade3_dta.xml (No such file or directory)
  	at java.base/java.io.FileInputStream.open0(Native Method)
  	at java.base/java.io.FileInputStream.open(Unknown Source)
  	at java.base/java.io.FileInputStream.<init>(Unknown Source)
  	at java.base/java.io.FileReader.<init>(Unknown Source)
  	at beast.base.util.FileUtils.load(Unknown Source)
  	at beast.base.parser.XMLParser.parseFile(Unknown Source)
  	at beastfx.app.beast.BeastMCMC.parseArgs(Unknown Source)
  	at beastfx.app.beast.BeastMain.main(Unknown 

In [40]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta.xml").resolve()   # absolute: fixes the doubled path
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 90

# --- move the old sequence-only outputs aside ---
for f in ["H_clade_3.log", "H_clade_3.trees"]:
    p = XML.parent / f
    if p.exists():
        p.rename(XML.parent / ("OLD_seqonly_" + f))
        print(f"renamed {f} -> OLD_seqonly_{f}")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found — set `beast` to the full path of BEAST*/bin/beast")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML}")
print(f"chain : {total:,}" if total else "chain : ?")
print(f"seed  : {SEED}")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown = [], [], 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if any(k in low for k in ("trait", "beagle", "error", "exception", "states")) and shown < 25:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = int(parts[0]) / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-15:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("clade3_dta*")) + sorted(XML.parent.glob("Host*")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

renamed H_clade_3.log -> OLD_seqonly_H_clade_3.log
renamed H_clade_3.trees -> OLD_seqonly_H_clade_3.trees

beast : /Applications/BEAST 2.7.7/bin/beast
xml   : /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/clade3_dta.xml
chain : 10,000,000
seed  : 12345
------------------------------------------------------------------
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  AncestralStateTreeLikelihood(traitedtreeLikelihood.Host) uses BeerLikelihoodCore
    AlignmentFromTrait(Host): [taxa, patterns, sites] = [162, 1, 1]
  Writing file Host_tree_with_trait.trees
  [  0.0 min] state            0  posterior  -39016.5610 

KeyboardInterrupt: 

In [41]:
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path
import shutil

XML   = Path("beast/clade3/clade3_dta.xml").resolve()
OUT   = XML.parent / "clade3_dta_merged.xml"
REMAP = {"ailurid": "wild_felid"}      # single-sequence state folded in

raw  = XML.read_text()
root = ET.fromstring(raw)

for el in root.iter():
    tn, val = el.get("traitname"), el.get("value")
    if not val or not tn or tn == "date-forward":
        continue
    pairs = [p for p in val.split(",") if "=" in p]
    print(f"traitname={tn!r} — {len(pairs)} taxa")
    print("  before:", dict(Counter(p.rsplit('=',1)[1].strip() for p in pairs)))
    new_pairs = [f"{p.rsplit('=',1)[0]}={REMAP.get(p.rsplit('=',1)[1].strip(), p.rsplit('=',1)[1].strip())}"
                 for p in pairs]
    print("  after :", dict(Counter(p.rsplit('=',1)[1] for p in new_pairs)))
    raw = raw.replace(val, ",".join(new_pairs))

ET.fromstring(raw)          # fails loudly if the edit broke anything
OUT.write_text(raw)
print(f"\nwrote {OUT.name}")

traitname='date' — 162 taxa
  before: {'2012.896': 1, '2019.312': 1, '1992.787': 1, '1992.667': 1, '1992.000': 1, '2013.268': 1, '2013.230': 1, '2013.238': 1, '2013.164': 1, '2013.195': 1, '2004.675': 1, '2019.304': 1, '2015.964': 1, '2015.989': 1, '2015.912': 1, '2015.951': 2, '2016.322': 1, '2016.336': 1, '2016.301': 1, '2017.660': 1, '2016.973': 2, '2017.301': 1, '2017.167': 1, '2016.210': 1, '2023.030': 1, '2022.819': 1, '2022.805': 1, '2023.392': 1, '2016.361': 1, '2022.926': 1, '2013.748': 1, '2017.970': 1, '2021.041': 1, '2023.622': 1, '2016.500': 28, '2017.500': 27, '2004.500': 2, '2007.500': 3, '2018.500': 1, '2010.500': 2, '2014.500': 29, '2013.500': 10, '2012.500': 1, '2015.500': 23}
  after : {'2012.896': 1, '2019.312': 1, '1992.787': 1, '1992.667': 1, '1992.000': 1, '2013.268': 1, '2013.230': 1, '2013.238': 1, '2013.164': 1, '2013.195': 1, '2004.675': 1, '2019.304': 1, '2015.964': 1, '2015.989': 1, '2015.912': 1, '2015.951': 2, '2016.322': 1, '2016.336': 1, '2016.301': 1, 

In [42]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta_merged.xml").resolve()
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 120

# park the outputs from the crashed run — Host_tree_with_trait.trees is corrupt
crashed = XML.parent / "crashed_run"
moved = []
for f in XML.parent.glob("*"):
    if f.is_file() and (f.name.startswith("clade3_dta.") and f.suffix in (".log", ".trees")
                        or f.name.startswith("Host_tree_with_trait")):
        crashed.mkdir(exist_ok=True)
        shutil.move(str(f), crashed / f.name)
        moved.append(f.name)
print(f"moved to crashed_run/: {moved}" if moved else "no previous outputs to move")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing — run the remap cell first")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML.name}")
print(f"chain : {total:,}" if total else "chain : ?")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown, errors = [], [], 0, 0
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if "randomchoiceunnormalized" in low or "java.lang.Error" in line:
            errors += 1
            if errors <= 3:
                print(f"  !! {line[:100]}")
            if errors == 4:
                print("  !! still crashing — kill this and switch to a symmetric trait model")
        elif any(k in low for k in ("trait", "beagle", "states", "writing file")) and shown < 20:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = int(parts[0]) / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {int(parts[0]):>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-12:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

moved to crashed_run/: ['clade3_dta.trees', 'clade3_dta.log', 'Host_tree_with_trait.trees']

beast : /Applications/BEAST 2.7.7/bin/beast
xml   : clade3_dta_merged.xml
chain : 10,000,000
------------------------------------------------------------------
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  AncestralStateTreeLikelihood(traitedtreeLikelihood.Host) uses BeerLikelihoodCore
    AlignmentFromTrait(Host): [taxa, patterns, sites] = [162, 1, 1]
  Writing file clade3_dta.log
  Writing file clade3_dta.trees
  Writing file Host_tree_with_trait.trees
  [  0.0 min] state            0  posterior  -39016.5610  ETA   0.0 min
  [  0.2

KeyboardInterrupt: 

In [43]:
from pathlib import Path
import hashlib, re
from collections import Counter

d = Path("beast/clade3")
for name in ["clade3_dta.xml", "clade3_dta_merged.xml"]:
    p = d / name
    raw = p.read_text()
    print(f"{name}: {hashlib.md5(raw.encode()).hexdigest()[:12]}  {len(raw):,} chars")
    print("   ailurid occurrences:", raw.count("ailurid"))

# where do the trait states actually live?
raw = (d / "clade3_dta.xml").read_text()
for m in re.finditer(r'<(\w+)[^>]*traitname="([^"]+)"', raw):
    print(f"\nelement <{m.group(1)}> traitname={m.group(2)!r}")

clade3_dta.xml: 8212e81f8433  374,522 chars
   ailurid occurrences: 6
clade3_dta_merged.xml: 8212e81f8433  374,522 chars
   ailurid occurrences: 6

element <trait> traitname='date'

element <traitSet> traitname='discrete'


In [44]:
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

XML   = Path("beast/clade3/clade3_dta.xml").resolve()
OUT   = XML.parent / "clade3_dta_merged.xml"
REMAP = {"ailurid": "wild_felid"}

tree = ET.parse(XML); root = tree.getroot()
changed = 0
for el in root.iter():
    tn = el.get("traitname")
    if not tn or tn.startswith("date"):
        continue
    src = "attr" if el.get("value") else "text"
    val = el.get("value") or (el.text or "")
    pairs = [x.strip() for x in val.replace("\n", " ").split(",") if "=" in x]
    if not pairs:
        continue
    print(f"<{el.tag}> traitname={tn!r} source={src} taxa={len(pairs)}")
    print("  before:", dict(Counter(x.rsplit('=',1)[1].strip() for x in pairs)))
    new = [f"{x.rsplit('=',1)[0].strip()}="
           f"{REMAP.get(x.rsplit('=',1)[1].strip(), x.rsplit('=',1)[1].strip())}" for x in pairs]
    print("  after :", dict(Counter(x.rsplit('=',1)[1] for x in new)))
    if src == "attr":
        el.set("value", ",".join(new))
    else:
        el.text = "\n" + ",\n".join(new) + "\n"
    changed += 1

if not changed:
    raise RuntimeError("no discrete traitset found — paste me the <traitSet> block")
tree.write(OUT, encoding="unicode", xml_declaration=True)
print(f"\nchanged {changed} element(s) -> {OUT.name}")
print("ailurid remaining:", OUT.read_text().count("=ailurid"))

<traitSet> traitname='discrete' source=text taxa=162
  before: {'procyonid': 91, 'wild_felid': 7, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23, 'ailurid': 1}
  after : {'procyonid': 91, 'wild_felid': 8, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23}

changed 1 element(s) -> clade3_dta_merged.xml
ailurid remaining: 0


In [45]:
from pathlib import Path
import xml.etree.ElementTree as ET
p = Path("beast/clade3/clade3_dta_merged.xml")
r = ET.fromstring(p.read_text())
print("data partitions:", len(r.findall(".//data")))
print("chainLength   :", [x.get("chainLength") for x in r.findall(".//run")])
print("loggers       :", [x.get("fileName") for x in r.findall(".//logger")])
print("BSSVS         :", "indicator" in p.read_text().lower())

data partitions: 2
chainLength   : ['10000000']
loggers       : ['clade3_dta.log', None, 'clade3_dta.trees', 'Host_tree_with_trait.trees']
BSSVS         : True


In [46]:
XML = Path("beast/clade3/clade3_dta_merged.xml").resolve()

In [47]:
import shutil, subprocess, time, re
from pathlib import Path

XML         = Path("beast/clade3/clade3_dta_merged.xml").resolve()
SEED        = 12345
THREADS     = 4
TIMEOUT_MIN = 120

# park outputs from the previous crashed attempt
crashed = XML.parent / "crashed_run2"
moved = []
for f in XML.parent.glob("*"):
    if f.is_file() and (f.name.startswith("clade3_dta.") and f.suffix in (".log", ".trees")
                        or f.name.startswith("Host_tree_with_trait")):
        crashed.mkdir(exist_ok=True)
        shutil.move(str(f), crashed / f.name)
        moved.append(f.name)
print(f"moved aside: {moved}" if moved else "no previous outputs to move")

def find_beast():
    exe = shutil.which("beast")
    if exe:
        return exe
    for root in [Path("/Applications"), Path.home() / "Applications", Path.home()]:
        if root.is_dir():
            for pattern in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(root.glob(pattern))
                if hits:
                    return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")
if not XML.is_file():
    raise FileNotFoundError(f"{XML} missing — run the remap cell first")

chain = re.search(r'chainLength="(\d+)"', XML.read_text())
total = int(chain.group(1)) if chain else None
print(f"\nbeast : {beast}")
print(f"xml   : {XML.name}")
print(f"chain : {total:,}" if total else "chain : ?")
print("-"*66)

cmd = [beast, "-seed", str(SEED), "-threads", str(THREADS), "-overwrite", str(XML)]
t0 = time.time()
proc = subprocess.Popen(cmd, cwd=str(XML.parent), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
posteriors, tail, shown, errors = [], [], 0, 0
passed_500k = False
try:
    for line in proc.stdout:
        line = line.rstrip(); tail.append(line); tail[:] = tail[-40:]
        low = line.lower()
        if "randomchoiceunnormalized" in low:
            errors += 1
            if errors <= 3:
                print(f"  !! trait error {errors} — {line[:80]}")
            if errors == 4:
                print("  !! SAME CRASH — kill this. Untick BSSVS in BEAUti, set Symmetric.")
        elif any(k in low for k in ("trait", "beagle", "states", "writing file")) and shown < 18:
            print(f"  {line[:100]}"); shown += 1
        parts = line.split()
        if len(parts) >= 2 and parts[0].isdigit():
            state = int(parts[0])
            if state > 500_000 and not passed_500k and errors == 0:
                passed_500k = True
                print("  >> past 500,000 states with no trait errors — the merge fixed it")
            try:
                posteriors.append(float(parts[1]))
                if len(posteriors) % 25 == 1 and total:
                    el = time.time() - t0
                    frac = state / total
                    eta = (el/frac - el)/60 if frac > 0.01 else 0
                    print(f"  [{el/60:5.1f} min] state {state:>12,}  "
                          f"posterior {parts[1]:>12}  ETA {eta:5.1f} min")
            except ValueError:
                pass
        if time.time() - t0 > TIMEOUT_MIN*60:
            proc.kill(); print(f"\nKILLED at {TIMEOUT_MIN} min"); break
finally:
    proc.wait()

print("-"*66)
print(f"exit {proc.returncode} after {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
if proc.returncode != 0:
    print("\nlast lines:\n  " + "\n  ".join(tail[-12:]))
elif posteriors:
    print(f"posterior {posteriors[0]:,.1f} -> {posteriors[-1]:,.1f}")

print("\noutput files:")
for f in sorted(XML.parent.glob("*.log")) + sorted(XML.parent.glob("*.trees")):
    kb = f.stat().st_size/1024
    print(f"  {f.name:<40} {kb:>10.1f} KB" + ("   <-- EMPTY" if kb < 1 else ""))

moved aside: ['clade3_dta.trees', 'clade3_dta.log', 'Host_tree_with_trait.trees']

beast : /Applications/BEAST 2.7.7/bin/beast
xml   : clade3_dta_merged.xml
chain : 10,000,000
------------------------------------------------------------------
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  Failed to load BEAGLE library: no hmsbeagle-jni in java.library.path: /usr/local/lib:/Applications/B
  AncestralStateTreeLikelihood(traitedtreeLikelihood.Host) uses BeerLikelihoodCore
    AlignmentFromTrait(Host): [taxa, patterns, sites] = [162, 1, 1]
  Writing file clade3_dta.log
  Writing file clade3_dta.trees
  Writing file Host_tree_with_trait.trees
  [  0.0 min] state            0  posterior  -39016.5595  ETA   0.0 min
  [  0.2 min] stat

KeyboardInterrupt: 

In [49]:
import shutil, subprocess, time, re
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

SRC         = Path("beast/clade3/clade3_dta_nobssvs.xml").resolve()
FINAL       = SRC.parent / "clade3_dta_final.xml"
SEEDS       = [12345, 54321]
THREADS     = 4
TIMEOUT_MIN = 120
REMAP       = {"ailurid": "wild_felid"}

# ---------- 1. merge the single-sequence state ----------
tree = ET.parse(SRC); root = tree.getroot()
changed = 0
for el in root.iter():
    tn = el.get("traitname")
    if not tn or tn.startswith("date"):
        continue
    src = "attr" if el.get("value") else "text"
    val = el.get("value") or (el.text or "")
    pairs = [y.strip() for y in val.replace("\n", " ").split(",") if "=" in y]
    if not pairs:
        continue
    print(f"<{el.tag}> traitname={tn!r} taxa={len(pairs)}")
    print("  before:", dict(Counter(y.rsplit('=',1)[1].strip() for y in pairs)))
    new = [f"{y.rsplit('=',1)[0].strip()}="
           f"{REMAP.get(y.rsplit('=',1)[1].strip(), y.rsplit('=',1)[1].strip())}" for y in pairs]
    print("  after :", dict(Counter(y.rsplit('=',1)[1] for y in new)))
    if src == "attr":
        el.set("value", ",".join(new))
    else:
        el.text = "\n" + ",\n".join(new) + "\n"
    changed += 1
if not changed:
    raise RuntimeError("no discrete traitset found")
tree.write(FINAL, encoding="unicode", xml_declaration=True)

# ---------- 2. verify ----------
raw = FINAL.read_text(); r = ET.fromstring(raw)
chain = re.search(r'chainLength="(\d+)"', raw)
total = int(chain.group(1)) if chain else None
print(f"\n{FINAL.name}")
print(f"  partitions : {len(r.findall('.//data'))}")
print(f"  chainLength: {total:,}" if total else "  chainLength: ?")
print(f"  BSSVS off  : {'indicator' not in raw.lower()}")
print(f"  ailurid    : {raw.count('=ailurid')} remaining")
print(f"  loggers    : {[x.get('fileName') for x in r.findall('.//logger') if x.get('fileName')]}")

# ---------- 3. park old outputs ----------
old = SRC.parent / "previous_runs"
for f in list(SRC.parent.glob("clade3_dta.*")) + list(SRC.parent.glob("Host_tree_with_trait*")):
    if f.is_file() and f.suffix in (".log", ".trees"):
        old.mkdir(exist_ok=True); shutil.move(str(f), old / f.name)

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")

# ---------- 4. run both chains ----------
for seed in SEEDS:
    print("\n" + "="*66); print(f"  CHAIN seed={seed}"); print("="*66)
    t0 = time.time(); errors = 0; posteriors = []; tail = []; ok500 = False
    proc = subprocess.Popen([beast, "-seed", str(seed), "-threads", str(THREADS),
                             "-overwrite", str(FINAL)],
                            cwd=str(FINAL.parent), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            line = line.rstrip(); tail.append(line); tail[:] = tail[-30:]
            if "randomchoiceunnormalized" in line.lower():
                errors += 1
                if errors <= 2: print(f"  !! trait error {errors}")
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                state = int(parts[0])
                if state > 500_000 and not ok500 and errors == 0:
                    ok500 = True; print("  >> 500k states clean")
                try:
                    posteriors.append(float(parts[1]))
                    if len(posteriors) % 50 == 1 and total:
                        el = time.time()-t0; frac = state/total
                        eta = (el/frac - el)/60 if frac > 0.01 else 0
                        print(f"  [{el/60:5.1f} min] {state:>12,}  {parts[1]:>12}  ETA {eta:5.1f} min")
                except ValueError:
                    pass
            if time.time()-t0 > TIMEOUT_MIN*60:
                proc.kill(); print("  KILLED on timeout"); break
    finally:
        proc.wait()

    print(f"  exit {proc.returncode} in {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
    if proc.returncode != 0:
        print("  last lines:\n    " + "\n    ".join(tail[-8:])); break

    for f in ["clade3_dta.log", "clade3_dta.trees", "Host_tree_with_trait.trees"]:
        p = FINAL.parent / f
        if p.exists():
            p.rename(FINAL.parent / f.replace(".", f"_seed{seed}.", 1))

print("\nfinal outputs:")
for f in sorted(FINAL.parent.glob("*seed*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

<traitSet> traitname='discrete' taxa=162
  before: {'procyonid': 91, 'wild_felid': 7, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23, 'ailurid': 1}
  after : {'procyonid': 91, 'wild_felid': 8, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23}

clade3_dta_final.xml
  partitions : 2
  chainLength: 10,000,000
  BSSVS off  : False
  ailurid    : 0 remaining
  loggers    : ['clade3_dta.log', 'clade3_dta.trees', 'Host_tree_with_trait.trees']

  CHAIN seed=12345
  [  0.0 min]            0   -38953.1594  ETA   0.0 min
  [  0.3 min]       50,000   -11886.5946  ETA   0.0 min


KeyboardInterrupt: 

In [50]:
from pathlib import Path
import re
raw = Path("beast/clade3/clade3_dta_nobssvs.xml").read_text()
for kw in ["indicator", "SVSGeneralSubstitutionModel", "GeneralSubstitutionModel",
           "Symmetric", "Asymmetric", "BSSVS", "nonZeroRates"]:
    print(f"  {kw:<32} {raw.lower().count(kw.lower())}")
print("\nlines mentioning indicator or SubstitutionModel:")
for l in raw.splitlines():
    if re.search(r"indicator|SubstitutionModel", l, re.I):
        print("   ", l.strip()[:120])

  indicator                        9
  SVSGeneralSubstitutionModel      2
  GeneralSubstitutionModel         2
  Symmetric                        0
  Asymmetric                       0
  BSSVS                            2
  nonZeroRates                     1

lines mentioning indicator or SubstitutionModel:
    <?xml version="1.0" encoding="UTF-8" standalone="no"?><beast beautitemplate='Standard' beautistatus='' namespace="beast.
    <stateNode id="rateIndicator.s:Host" spec="parameter.BooleanParameter" dimension="15">true</stateNode>
    <arg idref="rateIndicator.s:Host"/>
    <substModel id="svs.s:Host" spec="beastclassic.evolution.substitutionmodel.SVSGeneralSubstitutionModel" rateIndicator="@
    <operator id="indicatorFlip.s:Host" spec="operator.BitFlipOperator" parameter="@rateIndicator.s:Host" weight="30.0"/>
    <operator id="BSSVSoperator.c:Host" spec="beastclassic.evolution.operators.BitFlipBSSVSOperator" indicator="@rateIndicat
    <log idref="rateIndicator.s:Host"/>
    <lo

In [51]:
from pathlib import Path
from collections import Counter

src = Path("data/processed/H_clade_3.fasta")
out = Path("data/processed/H_clade_3_5state.fasta")

lines, hosts = [], []
for l in src.read_text().splitlines():
    if l.startswith(">"):
        acc, host, year = l[1:].split()[0].split("|")
        if host == "ailurid":
            host = "wild_felid"      # single observation folded in
        hosts.append(host)
        lines.append(f">{acc}|{host}|{year}")
    elif l.strip():
        lines.append(l.strip())
out.write_text("\n".join(lines) + "\n")

print(f"{len(hosts)} sequences -> {out.name}")
print(dict(Counter(hosts)))
print("states:", len(set(hosts)), "-> rateIndicator dimension should be",
      len(set(hosts))*(len(set(hosts))-1)//2, "if symmetric")

162 sequences -> H_clade_3_5state.fasta
{'procyonid': 91, 'wild_felid': 8, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23}
states: 5 -> rateIndicator dimension should be 10 if symmetric


In [52]:
from pathlib import Path
d = Path("data/processed")
print("in data/processed:")
for f in sorted(d.glob("*.fasta")):
    print(f"  {f.name:<40} {f.stat().st_size/1024:>8.1f} KB")

in data/processed:
  H_aligned.fasta                            3530.0 KB
  H_clade_3.fasta                             324.5 KB
  H_clade_3_5state.fasta                      324.5 KB
  H_clade_3_clean.fasta                       318.3 KB
  H_global400.fasta                           801.8 KB
  H_global400_clean.fasta                     801.4 KB
  H_labelled.fasta                           2558.5 KB
  sequences_H.fasta                          2526.5 KB
  sequences_all.fasta                        5527.1 KB


In [54]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

p = Path("beast/clade3/clade3_5state.xml")
raw = p.read_text(); r = ET.fromstring(raw)

print("partitions   :", len(r.findall(".//data")))
print("chainLength  :", [x.get("chainLength") for x in r.findall(".//run")])
print("rateIndicator:", raw.count("rateIndicator"), "(want 0)")
import re
m = re.search(r'rateIndicator[^>]*dimension="(\d+)"', raw)
print("  dimension  :", m.group(1) if m else "none — BSSVS is off")
print("loggers      :", [x.get("fileName") for x in r.findall(".//logger") if x.get("fileName")])

for el in r.iter():
    tn = el.get("traitname")
    if tn and not tn.startswith("date"):
        val = el.get("value") or (el.text or "")
        pairs = [y.strip() for y in val.replace("\n"," ").split(",") if "=" in y]
        print("trait states :", dict(Counter(y.rsplit('=',1)[1] for y in pairs)))

partitions   : 2
chainLength  : ['10000000']
rateIndicator: 3 (want 0)
  dimension  : 10
loggers      : ['clade3_5state.log', 'clade3_5state.trees', 'clade3_5state.host.trees']
trait states : {'procyonid': 91, 'wild_felid': 8, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23}


In [55]:
from pathlib import Path
raw = Path("beast/clade3/clade3_5state.xml").read_text()
for kw in ["indicatorFlip", "BSSVSoperator", "BitFlip", "nonZeroRates"]:
    print(f"  {kw:<20} {raw.count(kw)}")
print("\nrateIndicator lines:")
for l in raw.splitlines():
    if "rateIndicator" in l:
        print("   ", l.strip()[:130])

  indicatorFlip        0
  BSSVSoperator        0
  BitFlip              0
  nonZeroRates         0

rateIndicator lines:
    <rateIndicator id="rateIndicator.s:host" spec="parameter.BooleanParameter" dimension="10" estimate="false">true</rateIndicator>


In [56]:
import shutil, subprocess, time, re
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

XML         = Path("beast/clade3/clade3_5state.xml").resolve()
SEEDS       = [12345, 54321]
THREADS     = 4
TIMEOUT_MIN = 150

# ---------- verify before committing ----------
raw = XML.read_text(); r = ET.fromstring(raw)
chain = re.search(r'chainLength="(\d+)"', raw)
total = int(chain.group(1)) if chain else None
states = {}
for el in r.iter():
    tn = el.get("traitname")
    if tn and not tn.startswith("date"):
        val = el.get("value") or (el.text or "")
        pairs = [y.strip() for y in val.replace("\n", " ").split(",") if "=" in y]
        if pairs:
            states = dict(Counter(y.rsplit('=', 1)[1] for y in pairs))

print(f"{XML.name}")
print(f"  partitions  : {len(r.findall('.//data'))}")
print(f"  chainLength : {total:,}" if total else "  chainLength : ?")
print(f"  trait states: {len(states)} -> {states}")
print(f"  BSSVS ops   : {sum(raw.count(k) for k in ('indicatorFlip','BSSVSoperator','BitFlip'))} (want 0)")
assert len(states) == 5, "expected 5 trait states"
assert sum(raw.count(k) for k in ("indicatorFlip", "BSSVSoperator", "BitFlip")) == 0, "BSSVS still active"

# ---------- park old outputs ----------
old = XML.parent / "previous_runs"
for f in XML.parent.glob("clade3_5state*"):
    if f.is_file() and f.suffix in (".log", ".trees"):
        old.mkdir(exist_ok=True); shutil.move(str(f), old / f.name)

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    return None

beast = find_beast()
if not beast:
    raise RuntimeError("BEAST not found")

# ---------- run both chains ----------
for seed in SEEDS:
    print("\n" + "="*66); print(f"  CHAIN seed={seed}"); print("="*66)
    t0, errors, posteriors, tail = time.time(), 0, [], []
    marks = {300_000: False, 1_500_000: False}
    proc = subprocess.Popen([beast, "-seed", str(seed), "-threads", str(THREADS),
                             "-overwrite", str(XML)],
                            cwd=str(XML.parent), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout:
            line = line.rstrip(); tail.append(line); tail[:] = tail[-30:]
            if "randomchoiceunnormalized" in line.lower():
                errors += 1
                if errors <= 2: print(f"  !! trait error {errors} — model still unstable")
            parts = line.split()
            if len(parts) >= 2 and parts[0].isdigit():
                state = int(parts[0])
                for mark in marks:
                    if state > mark and not marks[mark] and errors == 0:
                        marks[mark] = True
                        print(f"  >> past {mark:,} clean "
                              f"({'first crash point' if mark == 300_000 else 'second crash point'})")
                try:
                    posteriors.append(float(parts[1]))
                    if len(posteriors) % 50 == 1 and total:
                        el = time.time() - t0; frac = state/total
                        eta = (el/frac - el)/60 if frac > 0.01 else 0
                        print(f"  [{el/60:5.1f} min] {state:>12,}  {parts[1]:>12}  ETA {eta:5.1f} min")
                except ValueError:
                    pass
            if time.time() - t0 > TIMEOUT_MIN*60:
                proc.kill(); print("  KILLED on timeout"); break
    finally:
        proc.wait()

    print(f"  exit {proc.returncode} in {(time.time()-t0)/60:.1f} min | trait errors: {errors}")
    if proc.returncode != 0:
        print("  last lines:\n    " + "\n    ".join(tail[-8:])); break

    for f in ["clade3_5state.log", "clade3_5state.trees", "clade3_5state.host.trees"]:
        p = XML.parent / f
        if p.exists():
            p.rename(XML.parent / f.replace(".", f"_seed{seed}.", 1))

print("\nfinal outputs:")
for f in sorted(XML.parent.glob("*seed*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

clade3_5state.xml
  partitions  : 2
  chainLength : 10,000,000
  trait states: 5 -> {'procyonid': 91, 'wild_felid': 8, 'wild_canid': 32, 'domestic_dog': 8, 'mustelid': 23}
  BSSVS ops   : 0 (want 0)

  CHAIN seed=12345
  [  0.0 min]            0   -38486.5912  ETA   0.0 min
  [  0.3 min]       50,000   -11341.6762  ETA   0.0 min
  [  0.6 min]      100,000    -9987.6442  ETA   0.0 min
  [  0.8 min]      150,000    -9821.3342  ETA  55.0 min
  [  1.1 min]      200,000    -9657.3917  ETA  54.6 min
  [  1.4 min]      250,000    -9648.1907  ETA  54.1 min
  [  1.7 min]      300,000    -9701.4674  ETA  53.6 min
  >> past 300,000 clean (first crash point)
  [  1.9 min]      350,000    -9654.8408  ETA  53.0 min
  [  2.2 min]      400,000    -9742.6317  ETA  52.7 min
  [  2.5 min]      450,000    -9673.4265  ETA  52.2 min
  [  2.7 min]      500,000    -9627.6028  ETA  51.7 min
  [  3.0 min]      550,000    -9693.1902  ETA  51.2 min
  [  3.3 min]      600,000    -9672.9327  ETA  50.9 min
  [  3.5 

In [58]:
from pathlib import Path
d = Path("beast/clade3")
for f in sorted(d.glob("*seed12345*")):
    print(f"  {f.name:<48} {f.stat().st_size/1024:>10.1f} KB")

  clade3_5state_seed12345.host.trees                  14177.6 KB
  clade3_5state_seed12345.log                           783.9 KB
  clade3_5state_seed12345.trees                       15729.4 KB


In [59]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

XML = Path("beast/clade3/clade3_5state.xml")
raw = XML.read_text()

print(f"{XML.name}\n")
print(f"UpDown occurrences : {raw.lower().count('updown')}")

# list every operator so we can see what's actually moving the tree and clock
print("\noperators:")
root = ET.fromstring(raw)
for op in root.findall(".//operator"):
    spec = op.get("spec", "")
    oid  = op.get("id", "")
    wt   = op.get("weight", "")
    targets = [op.get(k) for k in ("parameter", "tree", "up", "down") if op.get(k)]
    print(f"  {oid:<44} {spec:<28} w={wt:<6} {' '.join(targets)}")

# which operators touch the clock rate and tree height?
print("\noperators touching ucldMean:")
for op in

SyntaxError: invalid syntax (796188375.py, line 23)

In [60]:
from pathlib import Path
import xml.etree.ElementTree as ET
import re

XML = Path("beast/clade3/clade3_5state.xml")
raw = XML.read_text()

print(f"{XML.name}\n")
print(f"UpDown occurrences : {raw.lower().count('updown')}")

# list every operator so we can see what's actually moving the tree and clock
print("\noperators:")
root = ET.fromstring(raw)
for op in root.findall(".//operator"):
    spec = op.get("spec", "")
    oid  = op.get("id", "")
    wt   = op.get("weight", "")
    targets = [op.get(k) for k in ("parameter", "tree", "up", "down") if op.get(k)]
    print(f"  {oid:<44} {spec:<28} w={wt:<6} {' '.join(targets)}")

# which operators touch the clock rate and tree height?
print("\noperators touching ucldMean:")
for op in root.findall(".//operator"):
    if "ucldMean" in ET.tostring(op, encoding="unicode"):
        print(f"  {op.get('id')}  ({op.get('spec')}, weight {op.get('weight')})")

print("\noperators touching the tree:")
for op in root.findall(".//operator"):
    if op.get("tree") or "Tree.t" in ET.tostring(op, encoding="unicode"):
        print(f"  {op.get('id')}  ({op.get('spec')}, weight {op.get('weight')})")

clade3_5state.xml

UpDown occurrences : 2

operators:
  geoMuScaler.c:host                           ScaleOperator                w=3.0    @traitClockRate.c:host
  georateScaler.s:host                         ScaleOperator                w=30.0   @relativeGeoRates.s:host
  KappaScaler.s:H_clade_3_5state               AdaptableOperatorSampler     w=0.05   
  AVMNOperator.H_clade_3_5state                kernel.AdaptableVarianceMultivariateNormalOperator w=0.1    
  KappaScalerX.s:H_clade_3_5state              kernel.BactrianScaleOperator w=0.1    @kappa.s:H_clade_3_5state
  FrequenciesExchanger.s:H_clade_3_5state      AdaptableOperatorSampler     w=0.05   
                                                                            w=       
  FrequenciesExchangerX.s:H_clade_3_5state     operator.kernel.BactrianDeltaExchangeOperator w=0.1    
  gammaShapeScaler.s:H_clade_3_5state          AdaptableOperatorSampler     w=0.05   
                                                              

In [1]:
import re, shutil, subprocess, os, stat, shlex
from pathlib import Path

SRC       = Path("beast/clade3/clade3_5state.xml").resolve()
CHAIN     = 100_000_000
LOG_EVERY = 100_000          # keeps 1000 samples per file at 100M
SEEDS     = [12345, 54321]
THREADS   = 4
RUNDIR    = SRC.parent / "run100M"

def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError("BEAST not found")

beast = find_beast()

raw = SRC.read_text()
raw, n1 = re.subn(r'(chainLength=")\d+(")', rf'\g<1>{CHAIN}\g<2>', raw)
def fix(m):
    return m.group(0) if 'id="screenlog"' in m.group(0) else re.sub(
        r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', m.group(0))
raw, n2 = re.subn(r'<logger\b[^>]*>', fix, raw)
print(f"chainLength -> {CHAIN:,} ({n1} edit), logEvery -> {LOG_EVERY:,} ({n2} loggers)")

# each chain gets its own directory, so the fixed output filenames don't collide
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    (sd / SRC.name).write_text(raw)

lines = ["#!/bin/bash", "set -e"]
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    lines += [f'echo "=== seed {seed} started $(date) ==="',
              f'cd {shlex.quote(str(sd))}',
              f'{shlex.quote(beast)} -seed {seed} -threads {THREADS} -overwrite '
              f'{shlex.quote(str(sd / SRC.name))} > run.out 2>&1',
              f'echo "=== seed {seed} finished $(date) ==="']
sh = RUNDIR / "run_all.sh"
sh.write_text("\n".join(lines) + "\n")
sh.chmod(sh.stat().st_mode | stat.S_IEXEC)

# caffeinate stops the Mac sleeping; nohup detaches from the kernel
proc = subprocess.Popen(
    f"nohup caffeinate -i {shlex.quote(str(sh))} > {shlex.quote(str(RUNDIR/'driver.log'))} 2>&1 &",
    shell=True, cwd=str(RUNDIR))

print(f"\nlaunched, pid group detached")
print(f"driver log : {RUNDIR/'driver.log'}")
print(f"chain logs : {RUNDIR}/seed*/run.out")
print(f"\nEstimated ~9 h per chain, ~18 h total. Safe to close this notebook.")
print("Leave the laptop plugged in. Closing the lid still sleeps it — use")
print("System Settings > Lock Screen, or just leave it open.")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/batyanightingale/projects/cdv-phylodynamics/notebooks/beast/clade3/clade3_5state.xml'

In [2]:
import os
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("cwd:", Path.cwd())

cwd: /Users/batyanightingale/projects/cdv-phylodynamics


In [3]:
import os, re, shutil, subprocess, stat, shlex, sys
from pathlib import Path

# ---------- 1. get to the repo root ----------
ROOT = Path.cwd()
while not (ROOT / "scripts").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "scripts").is_dir():
    raise RuntimeError(f"repo root not found from {Path.cwd()} — set it manually")
os.chdir(ROOT)
print(f"repo root : {ROOT}")

# ---------- 2. environment report ----------
print(f"python    : {sys.executable}")
in_env = "cdv-phylo" in sys.executable
print(f"kernel env: {'cdv-phylo' if in_env else 'NOT cdv-phylo (base?)'}")
if not in_env:
    print("  ^ fine for launching BEAST (it is a separate Java app).")
    print("    Switch kernels tomorrow before any pipeline work.")

# ---------- 3. find BEAST ----------
def find_beast():
    exe = shutil.which("beast")
    if exe: return exe
    for rt in [Path("/Applications"), Path.home()/"Applications", Path.home()]:
        if rt.is_dir():
            for pat in ("BEAST*/bin/beast", "*/BEAST*/bin/beast"):
                hits = sorted(rt.glob(pat))
                if hits: return str(hits[0])
    raise RuntimeError("BEAST not found — set `beast` manually")

beast = find_beast()
print(f"beast     : {beast}")

# ---------- 4. build the 100M XMLs ----------
SRC       = (ROOT / "beast/clade3/clade3_5state.xml").resolve()
CHAIN     = 100_000_000
LOG_EVERY = 100_000
SEEDS     = [12345, 54321]
THREADS   = 4
RUNDIR    = SRC.parent / "run100M"

if not SRC.is_file():
    raise FileNotFoundError(f"{SRC} missing. In {SRC.parent}: "
        f"{sorted(p.name for p in SRC.parent.iterdir()) if SRC.parent.is_dir() else 'dir missing'}")

raw = SRC.read_text()
raw, n_chain = re.subn(r'(chainLength=")\d+(")', rf'\g<1>{CHAIN}\g<2>', raw)
def fix(m):
    return m.group(0) if 'id="screenlog"' in m.group(0) else re.sub(
        r'logEvery="\d+"', f'logEvery="{LOG_EVERY}"', m.group(0))
raw, n_log = re.subn(r'<logger\b[^>]*>', fix, raw)

print(f"\nchainLength -> {CHAIN:,}   ({n_chain} edit, want 1)")
print(f"logEvery    -> {LOG_EVERY:,}   ({n_log} loggers scanned, want 4)")
assert n_chain == 1, "chainLength not edited — check the XML"

for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    sd.mkdir(parents=True, exist_ok=True)
    (sd / SRC.name).write_text(raw)
print(f"run dirs   : {[p.name for p in sorted(RUNDIR.glob('seed*'))]}")

# ---------- 5. driver script ----------
lines = ["#!/bin/bash", "set -e"]
for seed in SEEDS:
    sd = RUNDIR / f"seed{seed}"
    lines += [f'echo "=== seed {seed} started $(date) ==="',
              f'cd {shlex.quote(str(sd))}',
              f'{shlex.quote(beast)} -seed {seed} -threads {THREADS} -overwrite '
              f'{shlex.quote(str(sd / SRC.name))} > run.out 2>&1',
              f'echo "=== seed {seed} finished $(date) ==="']
sh = RUNDIR / "run_all.sh"
sh.write_text("\n".join(lines) + "\n")
sh.chmod(sh.stat().st_mode | stat.S_IEXEC)

# ---------- 6. launch, detached ----------
subprocess.Popen(
    f"nohup caffeinate -i {shlex.quote(str(sh))} > {shlex.quote(str(RUNDIR/'driver.log'))} 2>&1 &",
    shell=True, cwd=str(RUNDIR))

print(f"\nLAUNCHED — detached from this notebook")
print(f"  driver log : {RUNDIR/'driver.log'}")
print(f"  chain output: {RUNDIR}/seed*/run.out")
print(f"\n~9 h per chain, ~18 h total. Safe to close Jupyter.")
print("Keep it plugged in and leave the lid open — caffeinate stops idle sleep,")
print("not lid-close sleep.")

repo root : /Users/batyanightingale/projects/cdv-phylodynamics
python    : /Users/batyanightingale/anaconda3/bin/python
kernel env: NOT cdv-phylo (base?)
  ^ fine for launching BEAST (it is a separate Java app).
    Switch kernels tomorrow before any pipeline work.
beast     : /Applications/BEAST 2.7.7/bin/beast

chainLength -> 100,000,000   (1 edit, want 1)
logEvery    -> 100,000   (4 loggers scanned, want 4)
run dirs   : ['seed12345', 'seed54321']

LAUNCHED — detached from this notebook
  driver log : /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/run100M/driver.log
  chain output: /Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/run100M/seed*/run.out

~9 h per chain, ~18 h total. Safe to close Jupyter.
Keep it plugged in and leave the lid open — caffeinate stops idle sleep,
not lid-close sleep.


In [4]:
from pathlib import Path
import time

RUNDIR = Path("/Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/run100M")
TOTAL = 100_000_000

drv = RUNDIR / "driver.log"
print(drv.read_text() if drv.exists() else "no driver log yet")
for sd in sorted(RUNDIR.glob("seed*")):
    log, out = sd / "clade3_5state.log", sd / "run.out"
    print(f"\n{sd.name}")
    if not out.exists():
        print("  not started yet"); continue
    if log.exists():
        rows = [l for l in log.read_text().splitlines()
                if l and not l.startswith("#") and l.split()[0].isdigit()]
        if rows:
            state = int(rows[-1].split()[0])
            print(f"  state {state:,} / {TOTAL:,}  ({state/TOTAL:.1%})")
            print(f"  last write {(time.time()-out.stat().st_mtime)/60:.0f} min ago")
        else:
            print("  log exists, no samples yet (normal in the first few minutes)")
    tail = out.read_text().splitlines()[-3:]
    print("  " + "\n  ".join(t[:90] for t in tail))

=== seed 12345 started Thu Aug 27 23:53:39 EDT 2026 ===


seed12345
  state 100,000 / 100,000,000  (0.1%)
  last write 0 min ago
           128000    -10202.7289     -9304.3429      -898.3859 5m52s/Msamples
           129000    -10215.2547     -9309.2004      -906.0543 5m52s/Msamples
           130000    -10216.3828     -9295.9225      -920.4603 5m52s/Msamples

seed54321
  not started yet


In [5]:
from pathlib import Path
import statistics, time

RUNDIR = Path("/Users/batyanightingale/projects/cdv-phylodynamics/beast/clade3/run100M")
TOTAL  = 100_000_000

def ess_rough(x):
    """Approximate ESS via autocorrelation. Indicative only — use Tracer for the real number."""
    n = len(x)
    if n < 10: return 0
    m, v = statistics.mean(x), statistics.pvariance(x)
    if v == 0: return 0
    s = 0.0
    for lag in range(1, min(n//3, 500)):
        c = sum((x[i]-m)*(x[i+lag]-m) for i in range(n-lag))/(n-lag)/v
        if c < 0.05: break
        s += c
    return n/(1+2*s)

drv = RUNDIR / "driver.log"
print(drv.read_text() if drv.exists() else "no driver log\n")

for sd in sorted(RUNDIR.glob("seed*")):
    print(f"--- {sd.name} ---")
    log, out = sd / "clade3_5state.log", sd / "run.out"
    if not log.is_file():
        print("  no log yet\n"); continue
    lines = [l for l in log.read_text().splitlines() if l and not l.startswith("#")]
    hdr = lines[0].split("\t")
    rows = [l.split("\t") for l in lines[1:] if l.split("\t")[0].isdigit()]
    if not rows:
        print("  no samples yet\n"); continue

    state = int(rows[-1][0])
    done = state >= TOTAL*0.999
    age = (time.time() - out.stat().st_mtime)/60 if out.exists() else None
    print(f"  {len(rows)} samples, last state {state:,} ({state/TOTAL:.1%})")
    print(f"  {'COMPLETE' if done else 'still running'}"
          + (f" — last write {age:.0f} min ago" if age is not None else ""))
    if not done and age is not None and age > 20:
        print("  !! no output for 20+ min — check it hasn't died")

    burn = len(rows)//10
    print(f"  (post 10% burn-in, {len(rows)-burn} samples)")
    for col in hdr:
        if any(k in col for k in ("posterior", "Tree.height", "ucldMean",
                                  "traitClockRate", "ucldStdev", "popSize")):
            i = hdr.index(col)
            try:
                vals = [float(r[i]) for r in rows[burn:]]
            except (ValueError, IndexError):
                continue
            e = ess_rough(vals)
            flag = "" if e >= 200 else "   <-- LOW"
            print(f"  {col:<34} mean {statistics.mean(vals):>12.5g}  ESS~{e:>6.0f}{flag}")

    for f in sorted(sd.glob("*")):
        if f.suffix in (".log", ".trees"):
            print(f"    {f.name:<36} {f.stat().st_size/1024:>9.1f} KB")
    print()

=== seed 12345 started Thu Aug 27 23:53:39 EDT 2026 ===
=== seed 12345 finished Fri Aug 28 08:08:56 EDT 2026 ===
=== seed 54321 started Fri Aug 28 08:08:56 EDT 2026 ===

--- seed12345 ---
  1001 samples, last state 100,000,000 (100.0%)
  COMPLETE — last write 177 min ago
  (post 10% burn-in, 901 samples)
  posterior                          mean      -9649.8  ESS~   438
  Tree.height                        mean       116.31  ESS~   346
  traitClockRate.host                mean     0.094357  ESS~   792
  popSize                            mean       141.52  ESS~   461
  ucldMean.H_clade_3_5state          mean   0.00052299  ESS~   494
  ucldStdev.H_clade_3_5state         mean      0.85048  ESS~   530
    clade3_5state.host.trees               14177.1 KB
    clade3_5state.log                        785.3 KB
    clade3_5state.trees                    15729.9 KB

--- seed54321 ---
  262 samples, last state 26,100,000 (26.1%)
  still running — last write 0 min ago
  (post 10% burn-in, 236 sa